# 모델 피팅 파이프라인 데모

노트북 하나 안에서 **원본 읽기 → 피처 → CV → 장부 → 최종 학습 → 제출 파일**까지 한 줄로 이어지게 짠 예시예요.
성능을 올리려는 노트북이 아니라 **틀**을 보여주려는 노트북이에요.

| 단계 | 셀 | 나중에 `src/`로 옮기면 |
|---|---|---|
| 0 설정 | `CFG` | `config.py` |
| 1 읽기 + 사상 ID | `load_raw`, `add_event_id` | `data.py` |
| 2 검증 | `assert` 묶음 | `data.py` |
| 3 fold | `make_folds` | `make_folds.py` |
| 4 피처 | `row_features`, `StationQ`, `FeaturePipe` | `features.py` |
| 5 모델 | `make_model` | `config.py` / `models.py` |
| 6 CV | `run_cv` → OOF 파일 + 장부 한 줄 | `run_cv.py` |
| 7 최종 학습·예측 | `fit_full_and_predict` | `train_full.py`, `predict.py` |
| 8 제출 검사 | `check_submission` | `predict.py` |

지켜야 하는 규칙은 세 가지예요.

1. **위에서 아래로 한 번에 돌아가야 해요.** 중간 셀만 다시 돌리거나 셀 순서를 바꿔야 결과가 나온다면 재현이 깨진 거예요. 제출 전에 `Restart & Run All`로 확인해요.
2. **바꾸는 곳은 설정 셀과 피처 셀뿐이에요.** CV 루프는 실험마다 고치지 않아요.
3. **관측소 통계처럼 train 분포를 쓰는 값은 학습 fold 안에서만 fit해요.** 이게 `fit` / `transform`을 나눠 둔 이유예요.

## 0. 설정

경로, seed, 모델 파라미터처럼 **실험마다 바뀔 수 있는 값을 전부 여기 모아요.**
아래 셀 곳곳에 숫자를 흩어 두면, 나중에 "그때 learning_rate가 뭐였지?"를 찾을 수 없어요.

결과물은 팀 공용 `oof/`, `ledger.csv`와 섞이지 않게 `cache/demo/` 아래에 따로 저장해요.

In [ ]:
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold

CFG = {
    "data_dir": Path("../data"),
    "out_dir": Path("cache/demo"),
    "seed": 42,
    "n_folds": 4,
    "owner": "한동",
    "model_params": dict(max_iter=300, learning_rate=0.1, max_leaf_nodes=63,
                         min_samples_leaf=50, early_stopping=False),
    "exclude_stations": ["ST0027"],
}
TARGET = "target_maxrise_6h"
TIDAL = ["ST0358", "ST0292", "ST0326", "ST0347", "ST0159", "ST0110", "ST0313", "ST0415",
         "ST0234", "ST0129", "ST0192", "ST0051", "ST0130", "ST0164", "ST0123"]

OUT = CFG["out_dir"]
for sub in ["oof", "models", "submissions"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

## 1. 원본 읽기 + 사상 ID

`../data/`의 CSV는 **읽기만 하고 절대 덮어쓰지 않아요.** 정렬·ID 같은 가공은 메모리 안에서만 해요.

사상 ID는 `AGENT_REFERENCE.md` 7장 방법 그대로예요. 전체 시각 축에서 1시간 넘게 비는 곳을 사상 경계로 봐요.
train과 test는 사상이 겹치지 않으니 **따로** 계산해요.

In [ ]:
def load_raw(name):
    return pd.read_csv(CFG["data_dir"] / f"{name}.csv")


def add_event_id(df):
    df = df.copy()
    ts = pd.to_datetime(df["timestamp"])
    uniq = pd.Series(np.sort(ts.unique()))
    starts = (uniq.diff() > pd.Timedelta("1h")).cumsum()
    df["event_id"] = ts.map(dict(zip(uniq, starts))).astype(int)
    return df


train = add_event_id(load_raw("train"))
test = add_event_id(load_raw("test"))
sample_sub = load_raw("sample_submission")
train.shape, test.shape, train.event_id.nunique(), test.event_id.nunique()

## 2. 검증

모델을 돌리기 전에 "데이터가 내가 생각한 모양인지"를 `assert`로 박아 둬요.
하나라도 틀리면 여기서 멈추니까, 엉뚱한 데이터로 30분 학습하는 일을 막아 줘요.

In [ ]:
assert train.row_id.is_unique and test.row_id.is_unique
assert TARGET in train and TARGET not in test
assert set(test.row_id) == set(sample_sub.row_id)
assert list(sample_sub.columns) == ["row_id", "pred_maxrise_6h"]
assert train[TARGET].notna().all()
print("검증 통과")

## 3. fold

`event_id` 기준 GroupKFold예요. 같은 사상의 행이 학습과 검증에 나뉘어 들어가면 점수가 부풀어요.

fold는 **한 번 만들어 파일로 저장하고, 그 뒤로는 파일을 읽어서** 써요.
그래야 모든 실험이 똑같은 분할 위에서 비교돼요. 팀 공용 `folds.csv`가 생기면 `fold_path`만 그 파일로 바꾸면 돼요.

ST0027은 학습에서 빼기로 했으니(EDA 결정) fold를 나누기 전에 빼요. 이 관측소는 예측을 0으로 둬요.

In [ ]:
def make_folds(df, path):
    if path.exists():
        return pd.read_csv(path)
    gkf = GroupKFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])
    fold = np.empty(len(df), dtype=int)
    for k, (_, va) in enumerate(gkf.split(df, groups=df.event_id)):
        fold[va] = k
    folds = df[["row_id", "event_id"]].assign(fold=fold)
    folds.to_csv(path, index=False)
    return folds


is_excluded = train.station_id.isin(CFG["exclude_stations"])
fit_df = train[~is_excluded].reset_index(drop=True)
folds = make_folds(fit_df, OUT / "folds.csv")
fit_df = fit_df.merge(folds[["row_id", "fold"]], on="row_id")
fit_df.groupby("fold").agg(rows=("row_id", "size"), events=("event_id", "nunique"))

fold별 행 수가 비슷한지 위 표로 확인해요. 155시간짜리 사상 하나가 행의 10%쯤이라 한 fold만 유독 커질 수 있어요.

## 4. 피처

피처를 두 종류로 나눠요. 이 구분이 노트북 파이프라인의 핵심이에요.

| 종류 | 예 | 계산에 필요한 것 | 누수 등급 |
|---|---|---|---|
| 행 피처 | `dwl_3h = wl_t − wl_t_minus_3h` | 그 행 하나 | A |
| 통계 피처 | `q` = 관측소 수위 분포에서 `wl_t`가 몇 분위인지 | **train 분포** | B |

행 피처는 그냥 함수 하나면 돼요. 통계 피처는 **`fit`(학습 데이터에서 분포를 외움)과 `transform`(외운 분포로 값 계산)을 나눠요.**
CV에서는 학습 fold로 fit하고, 최종 학습에서는 train 전체로 fit해요. test에는 그 fit 결과를 그대로 써요.

In [ ]:
RAW = [c for c in train.columns
       if c not in ["row_id", "timestamp", "station_id", "event_id", TARGET]]
WL_LAGS = {1: "wl_t_minus_1h", 3: "wl_t_minus_3h", 6: "wl_t_minus_6h",
           12: "wl_t_minus_12h", 24: "wl_t_minus_24h"}


def row_features(df):
    X = df[RAW].copy()
    for k, col in WL_LAGS.items():
        X[f"dwl_{k}h"] = df.wl_t - df[col]
    for src in ["imerg", "sfc"]:
        X[f"{src}_r1_6"] = df[f"{src}_rain_6h"] - df[f"{src}_rain_1h"]
        X[f"{src}_r6_24"] = df[f"{src}_rain_24h"] - df[f"{src}_rain_6h"]
    return X

In [ ]:
class StationQ:
    def fit(self, df):
        self.ref_ = {s: np.sort(g.to_numpy()) for s, g in df.groupby("station_id").wl_t}
        return self

    def transform(self, df):
        q = np.full(len(df), np.nan)
        wl = df.wl_t.to_numpy()
        for s, idx in df.groupby("station_id").indices.items():
            ref = self.ref_.get(s)
            if ref is not None:
                q[idx] = np.searchsorted(ref, wl[idx], side="right") / len(ref)
        return q

`FeaturePipe`는 위 두 개를 묶은 거예요. 실험마다 바꾸는 건 **어떤 피처를 켤지(`use_diff`, `use_q`)뿐**이에요.
학습 데이터에 없던 관측소는 `q`가 NaN이 되는데, HGB·LightGBM은 NaN을 그대로 받아요.

In [ ]:
class FeaturePipe:
    def __init__(self, use_diff=False, use_q=False):
        self.use_diff, self.use_q = use_diff, use_q

    def fit(self, df):
        if self.use_q:
            self.q_ = StationQ().fit(df)
        return self

    def transform(self, df):
        X = row_features(df) if self.use_diff else df[RAW].copy()
        if self.use_q:
            X["q"] = self.q_.transform(df)
            X["q_x_rain"] = X.q * np.log1p(df.sfc_rain_6h)
        return X

## 5. 모델

모델을 만드는 곳도 함수 하나로 모아요. LightGBM으로 바꿀 때는 이 함수 안만 바꾸면 돼요.

```python
import lightgbm as lgb
return lgb.LGBMRegressor(**CFG["model_params"], random_state=CFG["seed"], verbose=-1)
```

In [ ]:
def make_model():
    return HistGradientBoostingRegressor(**CFG["model_params"], random_state=CFG["seed"])


def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))

## 6. CV → OOF 파일 + 장부

`run_cv`가 이 노트북의 몸통이에요. fold마다 순서가 항상 같아요.

1. 학습 fold / 검증 fold로 나누고
2. **학습 fold로만** `pipe.fit`
3. 두 쪽 모두 `pipe.transform`
4. 모델 학습 → 검증 fold 예측 → `oof`에 채움

끝나면 OOF를 `oof/<실험ID>.csv`(`row_id, fold, y_true, oof`)로 저장해요. ST0027 행은 예측 0으로 붙여서 전체 train 기준 점수를 내요.
모델 비교와 앙상블은 나중에 이 파일만 있으면 돼요. 모델을 다시 돌릴 필요가 없어요.

In [ ]:
def run_cv(exp_id, pipe_kwargs):
    t0 = time.time()
    oof = np.zeros(len(fit_df))
    for k in range(CFG["n_folds"]):
        tr, va = fit_df[fit_df.fold != k], fit_df[fit_df.fold == k]
        pipe = FeaturePipe(**pipe_kwargs).fit(tr)
        model = make_model().fit(pipe.transform(tr), tr[TARGET])
        oof[va.index] = model.predict(pipe.transform(va))
        print(f"  fold {k}: {rmse(va[TARGET], oof[va.index]):.4f}")

    res = fit_df[["row_id", "fold"]].assign(y_true=fit_df[TARGET], oof=oof)
    excl = train.loc[is_excluded, ["row_id"]].assign(fold=-1, y_true=train.loc[is_excluded, TARGET], oof=0.0)
    res = pd.concat([res, excl], ignore_index=True)
    res.to_csv(OUT / "oof" / f"{exp_id}.csv", index=False)
    return res, time.time() - t0

### 장부 한 줄

ROADMAP 2.4의 장부 열을 그대로 채워요. 기준 실험(`base_id`)의 OOF 파일을 읽어서 fold별·사상별로 비교해요.

- **상위 5사상 제외 RMSE**: 여기서는 *행 수가 가장 많은(가장 긴) 사상 5개*를 빼고 계산했어요. 팀에서 정의가 다르면 `top5_events`만 바꾸면 돼요.
- 같은 실험ID로 다시 돌리면 장부의 그 줄을 **덮어써요**. 노트북을 처음부터 다시 돌려도 장부에 중복 줄이 쌓이지 않아요.

In [ ]:
event_of = train.set_index("row_id").event_id
top5_events = event_of.value_counts().index[:5]
nontidal_rows = set(train.row_id[~train.station_id.isin(TIDAL + CFG["exclude_stations"])])


def score_of(d, col="oof"):
    return rmse(d.y_true, d[col])


def ledger_row(exp_id, res, secs, change, base_id=None, memo=""):
    res = res.assign(event_id=res.row_id.map(event_of).to_numpy())
    row = {"실험ID": exp_id, "날짜": pd.Timestamp.now().strftime("%Y-%m-%d"), "담당": CFG["owner"],
           "바꾼 내용": change, "전체 OOF RMSE": round(score_of(res), 5),
           "비조석 RMSE": round(score_of(res[res.row_id.isin(nontidal_rows)]), 5),
           "상위 5사상 제외 RMSE": round(score_of(res[~res.event_id.isin(top5_events)]), 5),
           "실행 시간(초)": round(secs), "판정": "", "메모": memo}
    if base_id:
        base = pd.read_csv(OUT / "oof" / f"{base_id}.csv").set_index("row_id").oof
        se = pd.DataFrame({"fold": res.fold, "event_id": res.event_id,
                           "new": (res.y_true - res.oof) ** 2,
                           "base": (res.y_true - base.loc[res.row_id].to_numpy()) ** 2})
        by_fold = se[se.fold >= 0].groupby("fold")[["new", "base"]].mean()
        by_event = se.groupby("event_id")[["new", "base"]].mean()
        row["기준 대비 변화량"] = f"{np.sqrt(se.new.mean() / se.base.mean()) - 1:+.2%} (vs {base_id})"
        row["개선된 fold 수(4개 중)"] = int((by_fold.new < by_fold.base).sum())
        row["사상별 개선 비율"] = f"{(by_event.new < by_event.base).mean():.0%}"
    return row


def write_ledger(row):
    path = OUT / "ledger.csv"
    led = pd.read_csv(path) if path.exists() else pd.DataFrame()
    if len(led):
        led = led[led["실험ID"] != row["실험ID"]]
    led = pd.concat([led, pd.DataFrame([row])], ignore_index=True)
    led.to_csv(path, index=False, encoding="utf-8-sig")
    return led

### 실험 두 개 돌려 보기

- **E00 기준**: 원래 컬럼 그대로
- **E01**: 수위 차분(`dwl_k`), 강우 창 분해, `q`, `q × log1p(sfc_rain_6h)` 추가

파라미터와 seed, fold는 그대로 두고 피처만 바꿨어요(ROADMAP 2.2). 실험 하나는 셀 두 줄이에요.

In [ ]:
res0, secs0 = run_cv("E00", dict())
write_ledger(ledger_row("E00", res0, secs0, "원 컬럼만 (기준)"))

In [ ]:
res1, secs1 = run_cv("E01", dict(use_diff=True, use_q=True))
led = write_ledger(ledger_row("E01", res1, secs1, "+ dwl_k, 강우 창 분해, q, q×rain", base_id="E00"))
led

## 7. 최종 학습 · 예측

장부를 보고 고른 실험ID를 `FINAL`에 적어요. 최종 학습은 CV와 세 가지가 달라요.

1. `pipe.fit`을 **train 전체**로 다시 해요. `q`의 기준 분포도 전체 train이 돼요(ROADMAP 누수 검사 5번).
2. 모델도 train 전체로 학습해요. 반복 횟수는 CV에서 쓴 값 그대로예요.
3. test는 **같은 `pipe.transform`**으로 피처를 만들어요. train용 코드와 test용 코드를 따로 두지 않아요.

파이프와 모델은 같이 저장해요. 피처를 만드는 데 쓴 분포를 같이 저장해야 나중에 같은 예측을 다시 낼 수 있어요.

In [ ]:
FINAL = {"exp_id": "E01", "pipe_kwargs": dict(use_diff=True, use_q=True)}


def fit_full_and_predict(exp_id, pipe_kwargs):
    pipe = FeaturePipe(**pipe_kwargs).fit(fit_df)
    model = make_model().fit(pipe.transform(fit_df), fit_df[TARGET])
    joblib.dump({"pipe": pipe, "model": model, "cfg": CFG}, OUT / "models" / f"{exp_id}.joblib")

    pred = model.predict(pipe.transform(test))
    pred[test.station_id.isin(CFG["exclude_stations"]).to_numpy()] = 0.0
    return pd.DataFrame({"row_id": test.row_id, "pred_maxrise_6h": pred})


sub = fit_full_and_predict(**FINAL)

## 8. 제출 검사

제출 파일은 **`sample_submission`의 행 순서에 맞춰서** 저장해요. 그다음 형식을 한 번 더 `assert`로 확인해요.

In [ ]:
def check_submission(sub):
    sub = sample_sub[["row_id"]].merge(sub, on="row_id", how="left")
    assert len(sub) == len(sample_sub)
    assert (sub.row_id.to_numpy() == sample_sub.row_id.to_numpy()).all()
    assert sub.pred_maxrise_6h.notna().all()
    return sub


sub = check_submission(sub)
path = OUT / "submissions" / f"{FINAL['exp_id']}.csv"
sub.to_csv(path, index=False)
print(path, sub.shape)
sub.pred_maxrise_6h.describe()

## 정리

이 노트북을 돌리고 나면 `cache/demo/`에 이렇게 쌓여요.

```
cache/demo/
├── folds.csv             # 한 번 만들고 계속 재사용
├── oof/E00.csv, E01.csv  # 비교·앙상블은 이것만으로
├── ledger.csv            # 실험 1회 = 한 줄
├── models/E01.joblib     # pipe(분포) + 모델 + 설정
└── submissions/E01.csv
```

새 실험을 할 때 손대는 곳은 정해져 있어요.

- **피처를 추가하면** `row_features`나 `FeaturePipe`를 고치고 `run_cv("E02", ...)` 한 줄을 추가해요.
- **모델이나 파라미터를 바꾸면** `CFG["model_params"]`나 `make_model`을 고쳐요. 기준이 바뀌는 거니까 기준 실험도 새 설정으로 다시 돌려요.
- **앙상블은** `oof/*.csv`를 읽어 가중 평균한 뒤 RMSE를 보면 돼요. 모델을 다시 학습할 필요가 없어요.

이 노트북이 길어지면 `FeaturePipe`, `run_cv` 같은 함수를 `src/features.py`, `src/cv.py`로 옮기고 노트북에서는 `import`만 해요.
노트북은 "설정 바꾸고 실행하고 결과 보는 곳"으로 남기는 거예요. 세 사람이 같은 함수를 쓰게 되는 것도 이때부터예요.